# Earth–Moon Pair Test of the Summed Pull

## Question

The [Closure Theorem](https://willrg.com/documents/WILL_RG_I.pdf#sec:closure) states that the Pull of a relation is the sum of both participants' readings, while the Push is one shared value:

$$\kappa_{AB}^2 + \kappa_{BA}^2 = 2\beta_{AB}^2$$

For two bodies at separation $r$, each reading is the partner's Schwarzschild radius over the separation, so the relation reads

$$2\beta_{AB}^2 = \frac{R_{s,A} + R_{s,B}}{r}$$

This notebook tests the summed Pull on the Earth–Moon relation. The Moon's orbit around Earth must be governed by $R_{s,E} + R_{s,M}$. A single-valued Pull would give $R_{s,E}$ alone. The two predictions differ by $R_{s,M}/R_{s,E} \approx 1.2\%$.

## Avoiding the fitted-parameter trap

A single fit of the Moon's orbit returns one total and absorbs the 1.2% into it. The test therefore uses three independent relations, each read separately:

1. Earth and its artificial satellites give $R_{s,E}$. A satellite's own Schwarzschild radius is negligible, so the fitted value is Earth's reading alone.
2. The Moon and its orbiters give $R_{s,M}$. For the same reason, the fitted value is the Moon's reading alone.
3. The Moon's orbit around Earth gives the pair total, computed here only from the measured sidereal month and the laser-measured mean distance.

No relation receives a fitted value from another. The satellite and orbiter fits model the third body only through the difference of its Pull at the orbiting body and at the central body. Relative to the central body's own pull, this difference is

$$\varepsilon = 2\,\frac{R_{s,\mathrm{third}}}{R_{s,\mathrm{central}}}\left(\frac{r_{\mathrm{orbit}}}{d}\right)^3$$

where $d$ is the Earth–Moon distance. It bounds how far an error in the assumed third-body value can shift the fitted reading. The script evaluates it for LAGEOS around Earth and for GRAIL around the Moon; both values lie orders of magnitude below 1.2%.

## Method

**Step 1. Schwarzschild radii.** Each catalogue value $GM$ is converted by $R_s = 2GM/c^2$. In relational terms, $GM$ is half of $c^2 R_s$, read from the orbital Push and separation of that relation.

**Step 2. Eccentricity.** On an ellipse the Moon spends more time far out than close in, so the time-averaged distance exceeds the semi-major axis. Averaging over one orbit gives

$$\langle r \rangle = a\left(1 + \tfrac{1}{2}e^2\right)$$

**Step 3. The Sun's three-point contribution.** The Sun's Pull at the Moon differs from its Pull at Earth. Averaged over one month, this difference acts outward on the Moon, so the Moon sits farther out than the pair alone would hold it at the measured period. To first order,

$$a = a_0\left(1 - \tfrac{1}{6}m^2\right), \qquad m = \frac{T_{\mathrm{month}}}{T_{\mathrm{year}}}$$

where $a_0$ is the separation the pair alone would hold at the measured period. Higher-order solar terms are neglected.

**Step 4. Closure with the summed Pull.** At separation $a_0$ the Push is $\beta = \nu a_0 / c$ with $\nu = 2\pi/T_{\mathrm{month}}$, and the summed Pull is $\kappa^2 = R_{s,\mathrm{pair}}/a_0$. Closure $\kappa^2 = 2\beta^2$ gives the pair total measured from the orbit:

$$R_{s,\mathrm{pair}} = \frac{2\,\nu^2 a_0^3}{c^2}$$

**Step 5. Comparison.** $R_{s,\mathrm{pair}}$ is compared with $R_{s,E} + R_{s,M}$ and with $R_{s,E}$ alone.

## Data

| Quantity | Value | Source | Origin |
| --- | --- | --- | --- |
| $GM_E$ | 398600.4418 km³/s² | IERS Conventions (2010), Table 1.1 ([TN36](https://iers-conventions.obspm.fr/content/tn36.pdf)) | Satellite laser ranging to LAGEOS-1 and LAGEOS-2 |
| $GM_M$ | 4902.80011526323 km³/s² | GRAIL gravity model GRGM1200A, PDS label [gggrx_1200a_shb_l50](https://pds-geosciences.wustl.edu/grail/grail-l-lgrs-5-rdr-v1/grail_1001/shbdr/gggrx_1200a_shb_l50.xml) | GRAIL inter-satellite ranging and Doppler tracking |
| Sidereal month | 27.321661 d | IERS / Astronomical Almanac | Measured period relative to the stars |
| Sidereal year | 365.256363 d | IERS / Astronomical Almanac | Measured period relative to the stars |
| Time-averaged Earth–Moon distance $\langle r\rangle$ | 385000.52899 km | Chapront & Francou (2003), A&A 404, 735, [ELP/MPP02](https://www.aanda.org/10.1051/0004-6361:20030529) | Constant term of the distance series, fitted to lunar laser ranges at centimetre level |
| Eccentricity $e$ | 0.054900489 | Chapront & Francou (2003), ELP/MPP02 | Fitted to the measured range variation |
| LAGEOS distance from Earth's centre | 12270 km | LAGEOS mission | Leakage bound only |
| GRAIL distance from the Moon's centre | 1790 km | GRAIL mission (about 50 km altitude) | Leakage bound only |

The ELP mean semi-major axis (384399 km) and the ELP constant $a_0$ (384747.98 km) are excluded, because both are computed from the pair total and would make the test circular. The difference between Earth-centred and barycentric time scales (a factor $1.55\times10^{-8}$) is far below the precision of this test and is ignored.

## Result

The pair total read from the Moon's orbit equals $R_{s,E} + R_{s,M}$ to within $2.5\times10^{-4}$ and exceeds $R_{s,E}$ alone by $1.26\%$. The measured orbit selects the summed Pull with a margin of about fifty to one. The residual $2.5\times10^{-4}$ is attributed to the neglected higher-order solar terms.


In [ ]:
# Earth-Moon pair test of the summed Pull (WILL RG Closure Theorem)
import math

c = 299792458.0          # speed of light, m/s (exact)
DAY = 86400.0            # s

# ---------------- Inputs ----------------
GM_E = 398600.4418e9         # m^3/s^2, IERS Conventions (2010) Table 1.1, SLR to LAGEOS
GM_M = 4902.80011526323e9    # m^3/s^2, GRAIL GRGM1200A (PDS label gggrx_1200a_shb_l50)
T_month = 27.321661 * DAY    # sidereal month, s
T_year = 365.256363 * DAY    # sidereal year, s
r_mean = 385000.52899e3      # time-averaged Earth-Moon distance, m (ELP/MPP02 constant term)
e = 0.054900489              # eccentricity of the mean lunar orbit (ELP/MPP02)
r_lageos = 12270e3           # LAGEOS distance from Earth's centre, m (leakage bound only)
r_grail = 1790e3             # GRAIL distance from the Moon's centre, m (leakage bound only)

# ---------------- Step 1: Schwarzschild radii ----------------
R_sE = 2.0 * GM_E / c**2
R_sM = 2.0 * GM_M / c**2
print("Step 1. Schwarzschild radii")
print(f"  R_s,E           = {R_sE:.9e} m")
print(f"  R_s,M           = {R_sM:.9e} m")
print(f"  R_s,M / R_s,E   = {R_sM / R_sE:.6e}   (size of the effect under test)")

# ---------------- Leakage bounds ----------------
eps_lageos = 2.0 * (R_sM / R_sE) * (r_lageos / r_mean)**3
eps_grail = 2.0 * (R_sE / R_sM) * (r_grail / r_mean)**3
print("\nLeakage of the assumed third-body value into each fitted reading")
print(f"  Moon into Earth's reading (LAGEOS)  = {eps_lageos:.2e}")
print(f"  Earth into Moon's reading (GRAIL)   = {eps_grail:.2e}")

# ---------------- Step 2: Eccentricity ----------------
a = r_mean / (1.0 + 0.5 * e**2)
print("\nStep 2. Semi-major axis from the time-averaged distance")
print(f"  a               = {a / 1e3:.3f} km")

# ---------------- Step 3: Sun's three-point contribution ----------------
m = T_month / T_year
a0 = a / (1.0 - m**2 / 6.0)
print("\nStep 3. Separation held by the pair alone at the measured period")
print(f"  m               = {m:.8f}")
print(f"  a0              = {a0 / 1e3:.3f} km")

# ---------------- Step 4: Closure with the summed Pull ----------------
nu = 2.0 * math.pi / T_month
beta = nu * a0 / c
R_s_pair = 2.0 * nu**2 * a0**3 / c**2
print("\nStep 4. Pair total read from the Moon's orbit")
print(f"  beta            = {beta:.9e}")
print(f"  R_s,pair        = {R_s_pair:.9e} m")
print(f"  closure check   = {(R_s_pair / a0) / (2.0 * beta**2):.12f}   (kappa^2 / 2 beta^2)")

# ---------------- Step 5: Comparison ----------------
R_s_sum = R_sE + R_sM
dev_sum = R_s_pair / R_s_sum - 1.0
dev_earth = R_s_pair / R_sE - 1.0
print("\nStep 5. Comparison")
print(f"  R_s,pair vs R_s,E + R_s,M   : relative difference = {dev_sum:+.2e}")
print(f"  R_s,pair vs R_s,E alone     : relative difference = {dev_earth:+.2e}")
print(f"  discrimination margin       : {abs(dev_earth) / abs(dev_sum):.0f} to 1")
